In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/many_states/"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01', 'HLA-A*01:01', 'HLA-A*03:01', 'HLA-B*07:02', 'HLA-B*27:05']

# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [4]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions

TARGET_LENGTHS= [9]
NUM_RUNS = 2

## 2.2 Train
### 2.2.1 Train 8-mer
#### 2.2.1.1 baum-welch

In [5]:
stop_threshold = 0
cutoff = 0.00005
per_symbol_portion = -22 / 9
sum_size = 0
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    sum_size += train_size
    stop_threshold +=  abs(per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS) * cutoff * sum_size
stop_threshold

26.739900000000002

In [6]:
ALLELES

['HLA-A*02:01', 'HLA-A*01:01', 'HLA-A*03:01', 'HLA-B*07:02', 'HLA-B*27:05']

In [7]:
%%time
algorithm = 'baum-welch'

per_name_models = dict()
per_allele_per_run_per_split_models=dict()
for allele_name in ALLELES:
    print(f"allele {allele_name}")
    per_allele_per_run_per_split_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        print(f"run {run_num}")

        print()

        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]

        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=40,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold,
                                        min_iterations=5
                                        )
        per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


allele HLA-A*02:01
run 0

split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: 70430.22666435002	Time (s): 12.89
[2] Improvement: 1326.5883627742296	Time (s): 12.63
[3] Improvement: 1900.3104402276222	Time (s): 12.79
[4] Improvement: 2654.772602221812	Time (s): 13.52
[5] Improvement: 3257.8031602139818	Time (s): 14.87
[6] Improvement: 3385.785016840324	Time (s): 16.5
[7] Improvement: 3041.94177011773	Time (s): 14.5
[8] Improvement: 2525.8415018576197	Time (s): 16.4
[9] Improvement: 2105.3576440900797	Time (s): 14.09
[10] Improvement: 1873.948136428371	Time (s): 13.43
[11] Improvement: 1822.0254411534406	Time (s): 12.84
[12] Improvement: 1929.562221803586	Time (s): 13.11
[13] Improvement: 2190.632418139139	Time (s): 12.86
[14] Improvement: 2574.7126651178114	Time (s): 13.06
[15] Improvement: 2961.438835407491	Time (s): 13.48
[16] Improvement: 3137.735019342508	Time (s): 12.78
[17] Improvement: 2964.987532417639	Time 

# Viterbi step

In [8]:
per_allele_per_run_per_split_models[ALLELES[0]].keys()

dict_keys([0, 1])

In [9]:
per_allele_per_run_per_split_prepared_models = dict()
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()

        for split_num, model in per_allele_per_run_per_split_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [10]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=False,
                                        algorithm='viterbi',
                                        verbose=True,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=stop_threshold,
                                        min_iterations= 5,
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: -5880.219107513549	Time (s): 5.222
[2] Improvement: 787.9982421690365	Time (s): 6.467
[3] Improvement: 384.0312353045447	Time (s): 4.55
[4] Improvement: 131.5551186967641	Time (s): 6.4
[5] Improvement: 39.10617798753083	Time (s): 5.996
[6] Improvement: -25.48527433082927	Time (s): 4.044
Total Training Improvement: -4563.013607686502
Total Training Time (s): 36.2573
split 0
train size 24309
Shuffling for fairness 
Simple training: Train Length 24309, init_batch_size None
[1] Improvement: -7450.680235623033	Time (s): 3.901
[2] Improvement: 506.55200605955906	Time (s): 3.552
[3] Improvement: 409.9491949802032	Time (s): 3.883
[4] Improvement: 205.542510167812	Time (s): 3.762
[5] Improvement: 292.984597428469	Time (s): 3.51
[6] Improvement: 283.59152467979584	Time (s): 3.64
[7] Improvement: 236.8935498208739	Time (s): 3.965
[8] Improvement: 130.1536549141165	Time (s): 

In [11]:
per_name_new_models.keys()

dict_keys(['0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_plus_viterbi', '1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_plus_viterbi', '1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_plus_viterbi', '1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0_plus_viterbi', '1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_27_05-0_plus_viterbi', '1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_27_05-0_plus_viterbi'])

In [12]:
from hmm_visualization_methods import *

In [13]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
Wall time: 4.53 s


In [14]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
Wall time: 4min 58s


In [15]:
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0) (Delete zero edges)
Converting graph for model (1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0) (Delete zero edges)
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0) (Delete zero edges)
Converting graph for model (1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0) (Delete zero edges)
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (1_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0) (Delete zero edges)
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_27_05-0) (Delete zero edges)
new starti

In [16]:
TARGET_PATH_TO_RESULTS

'experiment_results/many_states/'

In [17]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
Wall time: 3.13 s


In [18]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
2
3
4
5
6
7
8
9
Wall time: 2min 54s


In [25]:
per_name_new_models.keys()

dict_keys(['0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_27_05-0_plus_viterbi', '0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0_plus_viterbi'])

In [22]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=2)

Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_03_01-0_plus_viterbi) (Delete zero edges)
new starting point
new starting point
new starting point
new starting point
new starting point
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01-0_plus_viterbi) (Delete zero edges)
new starting point
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_27_05-0_plus_viterbi) (Delete zero edges)
new starting point
Converting graph for model (0_run-9-40-st-baum-welch-alg-5-min_iters_model-HLA_B_07_02-0_plus_viterbi) (Delete zero edges)
new starting point
new starting point


In [62]:
model = per_name_models['0_run-9-10-40-st-baum-welch-alg-5-min_iters_model-HLA_A_02_01-0']

In [67]:
peptides = per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][10][:50]

In [69]:
for peptide in peptides:
    logp, path = model.viterbi(peptide)
    print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'LLFNILGGWV'  -- Log Probability: -30.897085708807133 -- Path: s031 s004 s017 s023 s010 s012 s021 s030 s035 s016
Viterbi for Sequence: 'TLYEHNNELV'  -- Log Probability: -30.174258369728715 -- Path: s011 s004 s017 s023 s010 s012 s002 s022 s026 s016
Viterbi for Sequence: 'DAVTYTEHAK'  -- Log Probability: -38.32822062561017 -- Path: s038 s000 s034 s028 s032 s019 s002 s022 s026 s006
Viterbi for Sequence: 'LLGVGNNLLV'  -- Log Probability: -28.79453839787274 -- Path: s031 s009 s025 s027 s015 s012 s021 s030 s035 s016
Viterbi for Sequence: 'YLIGGDVKSL'  -- Log Probability: -27.016003243469335 -- Path: s031 s004 s017 s028 s032 s018 s002 s007 s039 s016
Viterbi for Sequence: 'KLPEKLQAEV'  -- Log Probability: -27.749035748954768 -- Path: s011 s004 s017 s023 s010 s002 s022 s026 s024 s016
Viterbi for Sequence: 'TLRNTNPNFV'  -- Log Probability: -33.06197281224592 -- Path: s011 s004 s017 s023 s010 s012 s021 s030 s035 s016
Viterbi for Sequence: 'KGDALVVAQL'  -- Log Probability: -3

In [71]:
for peptide in peptides:
    logp, path = model.maximum_a_posteriori(list(peptide))
    print("Max a post for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path ) ))

Max a post for Sequence: 'LLFNILGGWV'  -- Log Probability: -4.194813896394699 -- Path: s031 s004 s017 s023 s008 s012 s021 s030 s035 s016
Max a post for Sequence: 'TLYEHNNELV'  -- Log Probability: -4.687950610946164 -- Path: s011 s004 s017 s023 s010 s012 s002 s022 s035 s016
Max a post for Sequence: 'DAVTYTEHAK'  -- Log Probability: -5.915240768861771 -- Path: s038 s000 s034 s027 s032 s019 s002 s022 s026 s006
Max a post for Sequence: 'LLGVGNNLLV'  -- Log Probability: -4.9860346832952835 -- Path: s031 s009 s025 s027 s015 s012 s021 s030 s035 s016
Max a post for Sequence: 'YLIGGDVKSL'  -- Log Probability: -3.861137682186069 -- Path: s031 s004 s017 s023 s032 s018 s002 s007 s039 s016
Max a post for Sequence: 'KLPEKLQAEV'  -- Log Probability: -5.62211216923685 -- Path: s011 s004 s017 s023 s010 s002 s022 s026 s024 s016
Max a post for Sequence: 'TLRNTNPNFV'  -- Log Probability: -5.1468730944201155 -- Path: s011 s004 s017 s023 s010 s012 s021 s030 s035 s016
Max a post for Sequence: 'KGDALVVAQL'  -

In [70]:
for peptide in peptides:
    logp = model.log_probability(peptide)
    print(" Sequence: '{}'  -- Log Probability: {} ".format(''.join( peptide ), logp, ))

 Sequence: 'LLFNILGGWV'  -- Log Probability: -27.35635784453421 
 Sequence: 'TLYEHNNELV'  -- Log Probability: -26.669716509387314 
 Sequence: 'DAVTYTEHAK'  -- Log Probability: -34.58069294800763 
 Sequence: 'LLGVGNNLLV'  -- Log Probability: -25.4491279006268 
 Sequence: 'YLIGGDVKSL'  -- Log Probability: -23.739935954318444 
 Sequence: 'KLPEKLQAEV'  -- Log Probability: -24.3796213417221 
 Sequence: 'TLRNTNPNFV'  -- Log Probability: -28.981879313425626 
 Sequence: 'KGDALVVAQL'  -- Log Probability: -27.401631532592777 
 Sequence: 'ALVLLILMTA'  -- Log Probability: -26.5205643654398 
 Sequence: 'VLAALVCYIV'  -- Log Probability: -27.028198435962654 
 Sequence: 'FLAEEGTAKL'  -- Log Probability: -22.91004149657301 
 Sequence: 'YISQFSYKEL'  -- Log Probability: -29.18135332232505 
 Sequence: 'KLRRGDLPFV'  -- Log Probability: -26.546691292883548 
 Sequence: 'YLREADFARV'  -- Log Probability: -26.816801350301947 
 Sequence: 'LLTSFTSQAV'  -- Log Probability: -25.878879078189577 
 Sequence: 'LLMIIGAL

In [72]:
peptides = per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9][:50]

In [73]:
for peptide in peptides:
    logp, path = model.viterbi(peptide)
    print("Viterbi for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path[1:-1] ) ))

Viterbi for Sequence: 'TLQSKDFKL'  -- Log Probability: -24.843949307986314 -- Path: s011 s004 s037 s023 s018 s021 s030 s039 s016
Viterbi for Sequence: 'LLLEVSEEL'  -- Log Probability: -22.586736166927015 -- Path: s031 s004 s017 s023 s008 s014 s022 s026 s016
Viterbi for Sequence: 'VVVKILKPV'  -- Log Probability: -27.85772417437325 -- Path: s031 s004 s037 s023 s010 s002 s007 s024 s016
Viterbi for Sequence: 'AVMQEVPLA'  -- Log Probability: -27.498095487722733 -- Path: s011 s004 s037 s023 s018 s002 s022 s035 s006
Viterbi for Sequence: 'GMASALEAL'  -- Log Probability: -24.14621706592266 -- Path: s011 s004 s037 s023 s010 s002 s022 s026 s016
Viterbi for Sequence: 'LLSIIFFPA'  -- Log Probability: -28.217037818536053 -- Path: s031 s004 s017 s029 s019 s021 s030 s035 s006
Viterbi for Sequence: 'IDKFVPQMP'  -- Log Probability: -35.05227726365399 -- Path: s038 s000 s034 s027 s001 s020 s036 s035 s006
Viterbi for Sequence: 'YIQKTLFKI'  -- Log Probability: -25.55994793216949 -- Path: s031 s004 s037 s0

In [74]:
for peptide in peptides:
    logp, path = model.maximum_a_posteriori(list(peptide))
    print("Max a post for Sequence: '{}'  -- Log Probability: {} -- Path: {}".format(''.join( peptide ), logp, " ".join( state.name for idx, state in path ) ))

Max a post for Sequence: 'TLQSKDFKL'  -- Log Probability: -3.0215446387905267 -- Path: s011 s004 s037 s023 s018 s005 s030 s039 s016
Max a post for Sequence: 'LLLEVSEEL'  -- Log Probability: -3.9786279833205036 -- Path: s031 s004 s017 s023 s008 s014 s022 s026 s016
Max a post for Sequence: 'VVVKILKPV'  -- Log Probability: -4.532896953537236 -- Path: s031 s004 s037 s023 s010 s002 s007 s035 s016
Max a post for Sequence: 'AVMQEVPLA'  -- Log Probability: -4.267698412846126 -- Path: s011 s004 s037 s023 s010 s002 s022 s035 s006
Max a post for Sequence: 'GMASALEAL'  -- Log Probability: -3.629362203923499 -- Path: s011 s004 s037 s023 s010 s002 s022 s026 s016
Max a post for Sequence: 'LLSIIFFPA'  -- Log Probability: -6.718725243841966 -- Path: s031 s004 s017 s027 s001 s020 s030 s035 s006
Max a post for Sequence: 'IDKFVPQMP'  -- Log Probability: -3.6019261637640767 -- Path: s038 s000 s034 s027 s001 s020 s022 s035 s006
Max a post for Sequence: 'YIQKTLFKI'  -- Log Probability: -2.262097506748866 -- 